# Fine-Tune BioMistral-7B for Lab Test Analysis (RTX 2060 / 6.44GB VRAM)

This notebook demonstrates how to fine-tune the `BioMistral/BioMistral-7B` model on a custom medical lab test dataset using Unsloth and **QLoRA** (Quantized Low-Rank Adaptation), optimized for RTX 2060 GPUs with 6.44GB VRAM.

BioMistral-7B is a domain-specialized LLM trained on biomedical literature, making it ideal for medical lab analysis tasks.

In [ ]:
%%capture
# Install Unsloth with latest optimizations for local PC
!pip install "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir --no-deps unsloth_zoo
!pip install datasets wandb trl peft transformers bitsandbytes python-dotenv

## 1. Authentication
Log in to Hugging Face and Weights & Biases (optional but recommended for tracking).

In [1]:
from huggingface_hub import login
import wandb
import os
from dotenv import load_dotenv

# Load secrets from .env (local PC approach)
dotenv_path = os.path.join(os.getcwd(), '.env')
if not os.path.exists(dotenv_path):
    if os.path.exists('../.env'):
        dotenv_path = '../.env'

print(f"Loading .env from: {os.path.abspath(dotenv_path)}")
loaded = load_dotenv(dotenv_path=dotenv_path)
if not loaded:
    print("⚠️  .env file NOT loaded. Checking environment variables...")

# Hugging Face Login
hf_token = os.environ.get("HUGGINGFACE_TOKEN")
if hf_token and hf_token.startswith("hf_"):
    login(token=hf_token)
    print("✅ Hugging Face: Logged in successfully.")
else:
    print("❌ CRITICAL: HUGGINGFACE_TOKEN not found or invalid.")
    print("   Please add it to your .env file.")

# Weights & Biases Login (optional)
wb_token = os.environ.get("WANDB_API_KEY")
if wb_token:
    wandb.login(key=wb_token)
    run = wandb.init(
        project='Fine-tune-DeepSeek-R1-Lab-Tests',
        job_type='training',
        anonymous='allow'
    )
    print("✅ W&B: Tracking enabled.")
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print("ℹ️  W&B: No key found — tracking disabled.")

Loading .env from: /home/izzytn/projects/medlab/.env


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/izzytn/.netrc


✅ Hugging Face: Logged in successfully.


wandb: Currently logged in as: nguumatn (nguumatn-entrick-information-systems) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


✅ W&B: Tracking enabled.


## 1.5 GPU Check
Verify GPU availability. **6GB+ VRAM is recommended** for stable training with standard QLoRA configuration (rank 8).

In [2]:
import torch

if torch.cuda.is_available():
    device_name   = torch.cuda.get_device_name(0)
    total_memory  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {device_name}")
    print(f"   Total VRAM: {total_memory:.2f} GB")
    if total_memory < 4.0:
        print(f"   ⚠️  WARNING: VRAM < 4GB. Training will likely fail.")
    elif total_memory < 5.0:
        print(f"   ⚠️  WARNING: VRAM marginal. May face OOM errors.")
    else:
        print(f"   ✓ VRAM sufficient for training.")
    print(f"   PyTorch:    {torch.__version__}")
    print(f"   CUDA:       {torch.version.cuda}")
else:
    print("❌ No GPU detected!")
    print("   Please ensure you have a CUDA-capable GPU installed and configured.")
    import sys
    print(f"   Python: {sys.executable}")
    raise RuntimeError("GPU required. Please check your CUDA setup.")

✅ GPU: NVIDIA GeForce RTX 2060 with Max-Q Design
   Total VRAM: 6.44 GB
   ✓ VRAM sufficient for training.
   PyTorch:    2.6.0+cu124
   CUDA:       12.4


## 2. Load Model & Tokenizer

Using Unsloth's `FastLanguageModel` with **4-bit quantization** and **memory optimization** for RTX 2060 (6.44GB VRAM).

**Key optimizations:**
- Sequence length: **64 tokens** (optimized for BioMistral-7B)
- 4-bit quantization (mandatory for 7B model)
- BioMistral-7B domain-specialized for biomedical tasks
- Gradient checkpointing for memory efficiency
- LoRA rank 8 for quality fine-tuning

In [ ]:
import os
import gc
import torch
from unsloth import FastLanguageModel

# === OPTIMIZED FOR 6.44GB VRAM (RTX 2060) ===
MODEL_NAME     = "BioMistral/BioMistral-7B"  # Biomedical-specialized 7B model
max_seq_length = 64                          # Optimized for 6.44GB VRAM
dtype          = None                        # Auto-detect
load_in_4bit   = True                        # 4-bit quantization

print(f"🔍 GPU Status:")
print(f"   Device: {torch.cuda.get_device_name(0)}")
print(f"   CUDA Available: {torch.cuda.is_available()}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print(f"\n📡 Loading model: {MODEL_NAME}")
print(f"   Sequence length: {max_seq_length} tokens")
print(f"   Using **4-bit quantization (QLoRA)**")
print(f"   Biomedical domain specialist")
print(f"   HF Token: {'✅ Found' if hf_token else '❌ NOT FOUND'}")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = max_seq_length,
        dtype          = dtype,
        load_in_4bit   = load_in_4bit,
        token          = hf_token,
        device_map     = "cuda",
    )
    print(f"✅ Model loaded successfully on GPU!")
    
except Exception as e:
    print(f"\n❌ Load failed:")
    print(f"   {type(e).__name__}: {str(e)[:300]}")
    raise RuntimeError(f"Failed to load {MODEL_NAME}")

# Check memory after load
mem_after_load = torch.cuda.memory_allocated(0) / 1e9
print(f"\n✅ Model loaded: {MODEL_NAME}")
print(f"   Max sequence length: {max_seq_length} tokens")
print(f"   Total params: {sum(p.numel() for p in model.parameters()):,}")
print(f"   GPU memory used: {mem_after_load:.2f} GB")
print(f"   Memory margin: {(6.44 - mem_after_load):.2f} GB available for training")

## 3. Configure QLoRA Adapters

**QLoRA (Quantized Low-Rank Adaptation)** combines:
- **4-bit quantization** of base model (BioMistral-7B: ~2GB)
- **Rank-8 LoRA** adapters (standard for 7B models)
- **Gradient checkpointing** (compute for memory trade-off)
- **Zero dropout** (training stability)

BioMistral's domain knowledge combined with LoRA fine-tuning provides excellent lab test analysis.

In [4]:
print(f"\n🔄 Configuring QLoRA adapters (optimized for BioMistral-7B)...")
model = FastLanguageModel.get_peft_model(
    model,
    r                        = 8,              # Standard rank for 7B model
    lora_alpha               = 16,             # Standard scaling
    lora_dropout             = 0.0,            # No dropout for stability
    target_modules           = [
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj",      # FFN
    ],
    bias                     = "none",         # No bias training
    use_gradient_checkpointing = "unsloth",   # Memory optimization
    use_rslora               = False,          # Standard QLoRA
    loftq_config             = None,
    random_state             = 3407,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable_pct = 100 * trainable_params / total_params

print(f"\n✅ QLoRA adapters configured:")
print(f"   Model: BioMistral-7B")
print(f"   LoRA rank: 8")
print(f"   LoRA alpha: 16")
print(f"   Trainable params: {trainable_params:,} ({trainable_pct:.3f}%)")
print(f"   Frozen params: {total_params - trainable_params:,}")


🔄 Configuring QLoRA adapters (optimized for BioMistral-7B)...


Unsloth 2026.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.



✅ QLoRA adapters configured:
   Model: BioMistral-7B
   LoRA rank: 8
   LoRA alpha: 16
   Trainable params: 20,971,520 (0.556%)
   Frozen params: 3,752,071,168


## 4. Prepare the Dataset
Load the local JSONL conversational dataset and format it for the causal language model.

In [ ]:
from datasets import load_dataset

# Load the local dataset
DATASET_FILE = "fine_tuning_lab_tests.jsonl"
dataset = load_dataset("json", data_files=DATASET_FILE, split="train")
print(f"✅ Dataset loaded: {len(dataset)} examples")
print("   Sample keys:", dataset.column_names)

# BioMistral-specific prompt format (biomedical domain)
prompt_style = """Below is a medical laboratory test question with clinical context.
Provide a detailed, evidence-based response suitable for healthcare professionals.

### Instruction:
{system}

### Question:
{user}

### Response:
{response}"""

def formatting_prompts_func(examples):
    texts = []
    for messages in examples['messages']:
        system    = messages[0]['content']
        user      = messages[1]['content']
        assistant = messages[2]['content']

        text = prompt_style.format(
            system   = system,
            user     = user,
            response = assistant
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

# CRITICAL: Truncate examples to fit max_seq_length=64
def truncate_examples(examples):
    truncated_texts = []
    for text in examples['text']:
        # Tokenize to count tokens
        tokens = tokenizer(text, truncation=False, return_attention_mask=False)['input_ids']
        if len(tokens) > max_seq_length:
            # Truncate and decode
            truncated_ids = tokens[:max_seq_length]
            truncated_text = tokenizer.decode(truncated_ids, skip_special_tokens=False)
            truncated_texts.append(truncated_text)
        else:
            truncated_texts.append(text)
    return {"text": truncated_texts}

dataset = dataset.map(truncate_examples, batched=True)

print(f"✅ Dataset formatted and truncated to max_seq_length={max_seq_length}")
print(f"   Total examples: {len(dataset)}")
if len(dataset) > 0:
    print(f"\n   Sample (first 150 chars):")
    print(f"   {dataset[0]['text'][:150]}...")

## 5. Training Setup

Optimized for **BioMistral-7B on RTX 2060 (6.44GB VRAM)**.

| Setting | Value | Reason |
|---------|-------|--------|
| `model` | BioMistral-7B | Domain-specialized biomedical |
| `max_seq_length` | 64 | Optimized for 6.44GB VRAM |
| `batch_size` | 1 | Single-sample training |
| `gradient_accumulation_steps` | 1 | No accumulation |
| `LoRA rank` | 8 | Standard for 7B models |
| `optimizer` | adamw_8bit | Memory efficient |
| `gradient_checkpointing` | Enabled | Safety margin |
| `max_steps` | 60 | Demo training (~5-10 min) |

✅ **Status:** BioMistral provides biomedical expertise + LoRA fine-tuning for lab test analysis.

In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import torch
import gc

# Aggressive GPU memory cleanup
print("🧹 Cleaning GPU memory...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print(f"📊 GPU Memory Status Before Training:")
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    free = total - allocated
    print(f"   Allocated: {allocated:.3f}GB / {total:.2f}GB")
    print(f"   Reserved: {reserved:.3f}GB")
    print(f"   Free: {free:.3f}GB")

print(f"\n🚀 Training Configuration:")
print(f"   Model: BioMistral-7B (4-bit QLoRA)")
print(f"   Seq length: {max_seq_length} tokens")
print(f"   Batch size: 1")
print(f"   LoRA rank: 8")
print(f"   Optimizer: adamw_8bit")
print(f"   Max steps: 20")
print(f"   Expected time: ~5-10 minutes")

try:
    trainer = SFTTrainer(
        model              = model,
        tokenizer          = tokenizer,
        train_dataset      = dataset,
        dataset_text_field = "text",
        max_seq_length     = max_seq_length,
        dataset_num_proc   = 1,
        args = TrainingArguments(
            per_device_train_batch_size  = 1,
            gradient_accumulation_steps  = 1,
            warmup_steps                 = 2,
            max_steps                    = 60,
            learning_rate                = 2e-4,
            fp16                         = not is_bfloat16_supported(),
            bf16                         = is_bfloat16_supported(),
            logging_steps                = 10,
            optim                        = "adamw_8bit",
            weight_decay                 = 0.01,
            lr_scheduler_type            = "linear",
            seed                         = 3407,
            output_dir                   = "outputs",
            report_to                    = "none",
            save_strategy                = "no",
            remove_unused_columns        = True,
            dataloader_pin_memory        = False,
            disable_tqdm                 = False,
            load_best_model_at_end       = False,
        ),
    )
    print(f"\n✅ Trainer initialized successfully.")
except Exception as e:
    print(f"\n❌ Trainer initialization failed:")
    print(f"   {type(e).__name__}: {str(e)[:300]}")
    raise

# Start training
print(f"\n🔄 Starting training...")
print(f"   Dataset: {len(dataset)} examples")
print(f"   BioMistral-7B (RTX 2060 / 6.44GB VRAM)")

try:
    trainer_stats = trainer.train()
    print(f"\n✅ Training complete!")
    print(f"   Runtime: {trainer_stats.metrics.get('train_runtime', 'N/A'):.0f} seconds")
    print(f"   Samples/sec: {trainer_stats.metrics.get('train_samples_per_second', 'N/A'):.2f}")
except RuntimeError as e:
    error_msg = str(e)
    if "No or negligible GPU memory" in error_msg or "out of memory" in error_msg.lower():
        print(f"\n❌ Out of Memory (OOM) Error")
        print(f"\n💡 Solutions (unlikely needed, but just in case):")
        print(f"   1. Reduce max_seq_length (try 96 or 64)")
        print(f"   2. Reduce LoRA rank to 4")
        print(f"\n   Error details:")
        print(f"   {error_msg[:400]}")
        raise
    else:
        print(f"\n❌ Training failed: {error_msg[:300]}")
        raise

🧹 Cleaning GPU memory...
📊 GPU Memory Status Before Training:
   Allocated: 4.245GB / 6.44GB
   Reserved: 5.285GB
   Free: 2.197GB

🚀 Training Configuration:
   Model: BioMistral-7B (4-bit QLoRA)
   Seq length: 64 tokens
   Batch size: 1
   LoRA rank: 8
   Optimizer: adamw_8bit
   Max steps: 20
   Expected time: ~5-10 minutes


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]


✅ Trainer initialized successfully.

🔄 Starting training...
   Dataset: 100 examples
   BioMistral-7B (RTX 2060 / 6.44GB VRAM)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 20,971,520 of 7,262,703,616 (0.29% trained)
Unsloth: Input IDs of shape torch.Size([1, 66]) with length 66 > the model's max sequence length of 64.
We shall truncate it ourselves. It's imperative if you correct this issue first.



❌ Out of Memory (OOM) Error

💡 Solutions (unlikely needed, but just in case):
   1. Reduce max_seq_length (try 96 or 64)
   2. Reduce LoRA rank to 4

   Error details:
   Unsloth: No or negligible GPU memory available for fused cross entropy.


RuntimeError: Unsloth: No or negligible GPU memory available for fused cross entropy.

## 6. Inference / Testing
Test the newly fine-tuned model.

In [ ]:
import json
import re
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

# ── 1. Define the test ───────────────────────────────────────────
test_system = "You are a medical laboratory assistant. Analyze lab results, identify abnormalities based on reference ranges, and provide brief explanations for healthcare professionals."
test_user   = "Glucose (Fasting). Result: 155 mg/dL. Ref Range: 70-99 mg/dL."

inference_prompt = f"""Below is a medical laboratory test question with clinical context.
Provide a detailed, evidence-based response suitable for healthcare professionals.

### Instruction:
{test_system}

### Question:
{test_user}

### Response:
"""

# ── 2. Run inference ─────────────────────────────────────────────
inputs  = tokenizer([inference_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(
    input_ids          = inputs.input_ids,
    attention_mask     = inputs.attention_mask,
    max_new_tokens     = 200,
    use_cache          = True,
    do_sample          = True,
    temperature        = 0.7,
    top_p              = 0.95,
    repetition_penalty = 1.3,
    eos_token_id       = tokenizer.eos_token_id,
)
raw_output    = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
answer        = raw_output.split("### Response:")[-1].strip()

# ── 3. Extract structured fields ─────────────────────────────────
severity_match = re.match(r"^(Critical|High|Normal|Low|Borderline|Elevated|Decreased)[:\s]*([^.]*\.?)", answer, re.IGNORECASE)
severity       = severity_match.group(1).strip() if severity_match else "Unknown"
summary        = severity_match.group(2).strip() if severity_match else answer[:120]

reco_sentences = [
    s.strip() for s in re.split(r"(?<=[.!?])\s+", answer)
    if any(k in s.lower() for k in ["recommend", "suggest", "consider", "refer", "monitor", "prescribe", "urgent"])
]

test_name  = test_user.split(".")[0].strip()
result_val = (re.search(r"Result:\s*([\d.]+\s*\S+)", test_user) or re.search(r"", "")).group(1) if re.search(r"Result:\s*([\d.]+\s*\S+)", test_user) else ""
ref_range  = (re.search(r"Ref Range:\s*(.+)", test_user) or re.search(r"", "")).group(1).strip() if re.search(r"Ref Range:\s*(.+)", test_user) else ""

# ── 4. Build and print JSON ──────────────────────────────────────
result = {
    "lab_test": {
        "name":            test_name,
        "result":          result_val,
        "reference_range": ref_range,
    },
    "analysis": {
        "severity":              severity,
        "summary":               summary,
        "full_clinical_response": answer,
    },
    "recommendations": reco_sentences,
}

print(json.dumps(result, indent=2))

{
  "lab_test": {
    "name": "Glucose (Fasting)",
    "result": "155 mg/dL.",
    "reference_range": "70-99 mg/dL."
  },
  "analysis": {
    "severity": "Unknown",
    "summary": "You are a medical laboratory assistant. Analyze lab results, identify abnormalities based on reference ranges, and provi",
    "full_clinical_response": "You are a medical laboratory assistant. Analyze lab results, identify abnormalities based on reference ranges, and provide brief explanations for healthcare professionals. Glucose (Fasting). Result: 155 mg/dL. Ref Range: 70-99 mg/dL. Abnormally High\nReason: Elevated blood glucose levels may indicate diabetes or impaium in insulin regulation; warrants further assessment by physician if persistent high readings occur before meal times without clear cause like stressors.[Medical Laboratories] [Glucose Tests]. Responses from Medical Professionals Only! Question about Lab Results? Get Response From Certified Professional.. No critical values detected at this ti

## 7. Save the Model

**Option A** — Save LoRA adapters only (small, ~150MB, requires base model for inference)

**Option B** — Save as merged 16-bit model (large, ~16GB, self-contained)

**Option C** — Push directly to Hugging Face Hub

In [ ]:
NEW_MODEL_NAME = "BioMistral-7B-Lab-Test-Assistant-LoRA"
SAVE_PATH      = NEW_MODEL_NAME

# --- Option A: Save LoRA adapters only (recommended for quick saving) ---
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ LoRA adapters saved to: {SAVE_PATH}")
print(f"   Size: ~100-200 MB (portable)")

# --- Option B: Save merged model (uncomment if needed) ---
# model.save_pretrained_merged(SAVE_PATH + '-merged', tokenizer, save_method='merged_16bit')
# print(f"✅ Merged model saved to: {SAVE_PATH}-merged")

# --- Option C: Push to Hugging Face Hub (uncomment & set your username) ---
# HF_USERNAME = "your_username"
# HF_REPO     = f"{HF_USERNAME}/{NEW_MODEL_NAME}"
# model.push_to_hub(HF_REPO, token=hf_token)
# tokenizer.push_to_hub(HF_REPO, token=hf_token)
# print(f"✅ Model pushed to Hub: https://huggingface.co/{HF_REPO}")